# EDA - Customer Lifetime Value Analysis (CLV)

# Ask

# Prepare

In [ ]:
# Import nescessary libraries for EDA

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv('data/raw/data.csv' , encoding="ISO-8859-1")

In [3]:
# Clean the formatting
data.to_csv('data/raw/data.csv', index=False, encoding='utf-8')

In [4]:
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [5]:
# The shape of data (rows and column)
data.shape

(541909, 8)

Data has 541909 rows and 8 columns

# Process

## Missing data

In [6]:
data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Missing data
- `Description`: 1454
- `CustormerID`: 135080 or 24.93% missing value (135080 / 135080 * 100 = 24.93%)

The significance missing value of `CustomerID` could lead to data bias or noise. To keep this clean, we will remove them.

In [7]:
data = data[data['CustomerID'].notna()]

In [8]:
# Check again after drop the null

data.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

## Duplicated Data

In [9]:
data.duplicated().sum()

np.int64(5225)

In [10]:
# Remove duplicate data

data.drop_duplicates(inplace=True)

## Converting Data type

In [11]:
data.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

`InvoiceDate` is currently string. Need to convert into date. `CustomerID` suppose to be a whole number which mean integer

In [12]:
# Convert string to datetime
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'])

In [13]:
# Convert float to integer
data['CustomerID'] = data['CustomerID'].astype(int, errors = 'ignore')

## Handle Cancelled transactions

In many e-commerce datasets, an InvoiceNo that starts with or is exactly 'C' represents a cancellation or a return.

In [14]:
# Filter out the 'C' which is 'Cancelled' order

data = data[data['InvoiceNo'].apply(lambda x : x[0]) != 'C']

# Analzye (EDA)

## Custormer Lifetime Value (CLV)

**CLV = ((Average Order Value x Purchase Frequency) / Churn Rate) x Profit Margin**

In [15]:
# Calculate total purchase

data['total_purchase'] = data['UnitPrice'] * data['Quantity']

In [16]:
data_group = data.groupby('CustomerID', as_index=False).agg({
  'total_purchase':'sum', 'InvoiceNo':'count', 'Country':'min', 'InvoiceDate':lambda date:(max(date) - min(date)).days}).rename(
    columns = {'InvoiceNo':'num_transactions', 'InvoiceDate':'days'}
  )

### 1. Calculate Average Order Value

In [36]:
data_group['avg_order_value'] = data_group['total_purchase'] / data_group['num_transactions']

In [18]:
data_group.head()

,CustomerID,total_purchase,num_transactions,Country,days,average_order_value
0,12346,77183.60,1,United Kingdom,0,77183.600000
1,12347,4310.00,182,Iceland,365,23.681319
2,12348,1797.24,31,Finland,282,57.975484
3,12349,1757.55,73,Italy,0,24.076027
4,12350,334.40,17,Norway,0,19.670588


### 2. Calculate Purchase Frequency

In [19]:
purchase_frequency = sum(data_group['num_transactions']) / data_group.shape[0]

### 3. Calculate Repeat Rate and Churn Rate 

In [ ]:
# Calculate Repeat Rate
repeat_rate = len(data_group[data_group['num_transactions'] >= 2]) / len(data_group)
round(repeat_rate * 100.0, 2)

98.34

The Percentage of Repeat Rate is `98.34%`

**Retention Churn (The "One-and-Done" Rate)**

*Churn Rate=1−Repeat Rate*

- When to use: Retail and E-commerce.
- What it tells you: "What percentage of my customers are 'One-Timers' who never came back?"

In [21]:
# Calculate Churn Rate
churn_rate = 1 - repeat_rate
round(churn_rate * 100.0, 2)

1.66

The Percentage of Churn Rate is `1.66%`

### 4. Calculate Profit Margin

Profit margin is the percentage of total sales that remains as profit. Assume our business has approximately 10% profit on the total sales

In [24]:
data_group['profit_margin'] = data_group['total_purchase'] * 0.1 

### 5. Calculate Custormer Lifetime Value (CLV)

In [ ]:
churn_rate

0.016593685180917306

In [37]:
# CLV = ((Average Order Value x Purchase Frequency) / Churn Rate) x Profit Margin

data_group['CLV'] = (data_group['avg_order_value'] * purchase_frequency) / churn_rate
data_group['CLV'] = data_group['CLV'] * 0.10 # profit margin
data_group['CLV'] = data_group['CLV'].apply(lambda x: '{:.2f}'.format(x))

In [38]:
data_group.head()

,CustomerID,total_purchase,num_transactions,Country,days,average_order_value,profit_margin,CLV,avg_order_value
0,12346,77183.60,1,United Kingdom,0,77183.600000,7718.360,42100652.22,77183.600000
1,12347,4310.00,182,Iceland,365,23.681319,431.000,12917.24,23.681319
2,12348,1797.24,31,Finland,282,57.975484,179.724,31623.37,57.975484
3,12349,1757.55,73,Italy,0,24.076027,175.755,13132.54,24.076027
4,12350,334.40,17,Norway,0,19.670588,33.440,10729.54,19.670588


## Customer Segmentation (DataViz)

In [ ]:
data_group_v1 = data_group